# DuckDB PyDough Database Connector

> ## 🚀 Initial Setup
>
> ---
>
> ### 1️⃣ DuckDB — Zero Infrastructure Required
>
> DuckDB is a **serverless, in-process** analytical database — no server, no credentials, no Docker container needed.
>
> By default it runs entirely **in memory**. You can also point it at a persistent file:
>
> | Mode | `database` value |
> |---|---|
> | In-memory (default) | `":memory:"` or omit the parameter |
> | Persistent file | `"path/to/mydb.duckdb"` |
>
> ---
>
> ### 2️⃣ Loading TPC-H Data
>
> DuckDB ships with a built-in TPC-H generator. Run the following once to populate your in-memory database with SF=1 TPC-H data:
>
> ```python
> import duckdb
> conn = duckdb.connect(":memory:")
> conn.execute("INSTALL tpch; LOAD tpch;")
> conn.execute("CALL dbgen(sf=1);")
> ```
>
> Then pass the existing connection to PyDough:
>
> ```python
> pydough.active_session.connect_database("duckdb", connection=conn)
> ```
>
> Alternatively, pass a file path and load the data yourself before connecting.

> ## 🔌 Installing the DuckDB Connector
>
> Make sure to have the **`duckdb`** package installed:
>
> ---
>
> - **If you're working inside the repo**:
> ```bash
> pip install -e ".[duckdb]"
> ```
>
> - **Or install the package directly**:
> ```bash
> pip install duckdb
> ```

> ## Importing Required Libraries
>
> ---

In [ ]:
import duckdb
import pydough
import datetime

> ## 🔑 Setting Up TPC-H Data and Connecting to DuckDB with PyDough
>
> ---
>
> ### 1️⃣ Create a DuckDB Connection and Load TPC-H Data
> - Open an in-memory DuckDB connection.
> - Use DuckDB's built-in `tpch` extension to generate SF=1 TPC-H tables in one call.
>
> ---
>
> ### 2️⃣ DuckDB-PyDough `connect_database()` Parameters
> - **`connection`** *(optional)*: A pre-existing `duckdb.DuckDBPyConnection` object. Use this when you have already loaded data into a connection (as shown here).
> - **`database`** *(optional)*: Path to a DuckDB database file, or `":memory:"` for an in-memory database. Ignored when `connection` is provided. Defaults to `":memory:"`.
>
> ---
>
> ### 3️⃣ Connect to DuckDB Using PyDough
> - `pydough.active_session.load_metadata_graph(...)`  
>   Loads the metadata graph describing the TPC-H schema (used for query planning).
> - `connect_database(...)`  
>   Registers the DuckDB connection with PyDough.

In [10]:
# Create an in-memory DuckDB connection and populate it with TPC-H SF=1 data
conn = duckdb.connect(":memory:")
conn.execute("INSTALL tpch; LOAD tpch;")
conn.execute("CALL dbgen(sf=1);")

# Load the TPC-H metadata graph
pydough.active_session.load_metadata_graph("../metadata/tpch_demo_graph.json", "TPCH")

# Connect PyDough to the pre-populated DuckDB connection
pydough.active_session.connect_database("duckdb", connection=conn)

DatabaseContext(connection=<pydough.database_connectors.database_connector.DatabaseConnection object at 0x11b991590>, dialect=<DatabaseDialect.DUCKDB: 'duckdb'>)

> ## ✨ Enabling PyDough's Jupyter Magic Commands
>
> ---
>
> This step loads the **`pydough.jupyter_extensions`** module, which adds custom magic commands (like `%%pydough`) to your notebook.
>
> ---
>
> ### 📊 What These Magic Commands Do
> - **Write PyDough directly** in notebook cells using:
>   ```python
>   %%pydough
>   ```
> - **Automatically render** query results inside the notebook.
>
> ---
>
> ### 💻 How It Works
> This is a **Jupyter-specific feature** — the `%load_ext` command dynamically loads these extensions into your **current notebook session**:
> ```python
> %load_ext pydough.jupyter_extensions
> ```

In [3]:
%load_ext pydough.jupyter_extensions

> ## 📊 Running TPC-H Query 1 with PyDough in DuckDB
>
> ---
>
> This cell runs **TPC-H Query 1** using **PyDough**.
>
> ---
>
> ### 📝 What the Query Does
> - **Computes summary statistics**: sums, averages, and counts for orders.
> - **Groups by**: `return_flag` and `line_status`.
> - **Filters by**: a shipping date cutoff.
>
> ---
>
> ### 📤 Output
> - `pydough.to_df(output)` converts the result to a **Pandas DataFrame**.
> - This makes it easy to inspect and analyze results directly in Python.
>
> ---

In [11]:
%%pydough
# TPCH Q1
output = (lines.WHERE((ship_date <= datetime.date(1998, 12, 1)))
        .PARTITION(name="groups", by=(return_flag, status))
        .CALCULATE(
            L_RETURNFLAG=return_flag,
            L_LINESTATUS=status,
            SUM_QTY=SUM(lines.quantity),
            SUM_BASE_PRICE=SUM(lines.extended_price),
            SUM_DISC_PRICE=SUM(lines.extended_price * (1 - lines.discount)),
            SUM_CHARGE=SUM(
                lines.extended_price * (1 - lines.discount) * (1 + lines.tax)
            ),
            AVG_QTY=AVG(lines.quantity),
            AVG_PRICE=AVG(lines.extended_price),
            AVG_DISC=AVG(lines.discount),
            COUNT_ORDER=COUNT(lines),
        )
        .ORDER_BY(L_RETURNFLAG.ASC(), L_LINESTATUS.ASC())
)

pydough.to_df(output)

,L_RETURNFLAG,L_LINESTATUS,SUM_QTY,SUM_BASE_PRICE,SUM_DISC_PRICE,SUM_CHARGE,AVG_QTY,AVG_PRICE,AVG_DISC,COUNT_ORDER
0,A,F,37734107.0,5.658655e+10,5.375826e+10,5.590907e+10,25.522006,38273.129735,0.049985,1478493
1,N,F,991417.0,1.487505e+09,1.413082e+09,1.469649e+09,25.516472,38284.467761,0.050093,38854
2,N,O,76633518.0,1.149352e+11,1.091896e+11,1.135610e+11,25.502020,38248.015609,0.050000,3004998
3,R,F,37719753.0,5.656804e+10,5.374129e+10,5.588962e+10,25.505794,38250.854626,0.050009,1478870


> ## 🔍 Inspecting the Generated SQL
>
> ---
>
> `pydough.to_sql(output)` returns the SQL query (translated for the DuckDB dialect) that PyDough generates for the query above, without executing it.

In [12]:
print(pydough.to_sql(output))

SELECT
  l_returnflag AS L_RETURNFLAG,
  l_linestatus AS L_LINESTATUS,
  COALESCE(SUM(l_quantity), 0) AS SUM_QTY,
  COALESCE(SUM(l_extendedprice), 0) AS SUM_BASE_PRICE,
  COALESCE(SUM(l_extendedprice * (
    1 - l_discount
  )), 0) AS SUM_DISC_PRICE,
  COALESCE(SUM(l_extendedprice * (
    1 - l_discount
  ) * (
    1 + l_tax
  )), 0) AS SUM_CHARGE,
  AVG(l_quantity) AS AVG_QTY,
  AVG(l_extendedprice) AS AVG_PRICE,
  AVG(l_discount) AS AVG_DISC,
  COUNT(*) AS COUNT_ORDER
FROM main.lineitem
WHERE
  l_shipdate <= CAST('1998-12-01' AS DATE)
GROUP BY
  1,
  2
ORDER BY
  1 NULLS FIRST,
  2 NULLS FIRST
